# 03 Product & Category Intelligence

**Research question:** How do purchasing patterns and customer review scores differ across product categories?
**Sources:** `analytics.fact_order_items` (item grain), `analytics.fact_reviews` (review grain, aggregated per order). **Filter:** all statuses; review avg over orders with a valid review score.

In [1]:
from src.db import get_connection

conn = get_connection(read_only=True)


In [2]:
# Aggregate review per order to avoid fanout when joining item-grain data
cat = conn.execute("""
WITH order_review AS (
  SELECT order_id, AVG(valid_review_score) AS avg_review
  FROM analytics.fact_reviews
  WHERE valid_review_score IS NOT NULL
  GROUP BY order_id
),
item_enriched AS (
  SELECT oi.order_id, oi.order_item_id,
         COALESCE(oi.product_category_name, 'unknown') AS category,
         oi.price, oi.freight_value
  FROM analytics.fact_order_items oi
)
SELECT category,
       COUNT(*) AS items,
       COUNT(DISTINCT i.order_id) AS orders,
       SUM(price) AS revenue,
       AVG(price) AS avg_item_price,
       AVG(r.avg_review) AS avg_review
FROM item_enriched i
LEFT JOIN order_review r ON i.order_id = r.order_id
GROUP BY category
ORDER BY revenue DESC
""").fetchdf()
cat.head(20)

,category,items,orders,revenue,avg_item_price,avg_review
0,beleza_saude,9670,8836,1258681.34,130.163531,4.141451
1,relogios_presentes,5991,5624,1205005.68,201.135984,4.019130
2,cama_mesa_banho,11115,9417,1036988.68,93.296327,3.902641
3,esporte_lazer,8641,7720,988048.97,114.344285,4.109828
4,informatica_acessorios,7827,6689,911954.32,116.513903,3.936554
5,moveis_decoracao,8334,6449,729762.49,87.564494,3.913296
6,cool_stuff,3796,3632,635290.85,167.357969,4.148577
7,utilidades_domesticas,6964,5884,632248.66,90.788148,4.057207
8,automotivo,4235,3897,592720.11,139.957523,4.066595
9,ferramentas_jardim,4347,3518,485256.46,111.630196,4.045159
